# dark-store-api Test Notebook

Interactive testing of the dark-store-api endpoints.

## Prerequisites

1. Start dark-ipfs cluster: `make -C ../../blockchain/dark-ipfs up`
2. Start the Store API: `docker compose up -d --build` or `uvicorn app.main:app --port 8003`

Current Store API behavior:

- `/health/live` is a lightweight liveness endpoint and does not touch IPFS/Cluster.
- `/health` is storage readiness and is cached by `IPFS_HEALTH_CACHE_TTL_SECONDS`.
- default write mode is `cluster_proxy`, so `POST /v1/store` writes through IPFS Cluster Proxy.


In [ ]:
import httpx
import json

BASE_URL = "http://localhost:8003"

## 1. Liveness and Readiness Checks


In [ ]:
live_response = httpx.get(f"{BASE_URL}/health/live")
print("Liveness status:", live_response.status_code)
print(json.dumps(live_response.json(), indent=2))
assert live_response.status_code == 200

readiness_response = httpx.get(f"{BASE_URL}/health", params={"refresh": "true"})
print("\nReadiness status:", readiness_response.status_code)
readiness = readiness_response.json()
print(json.dumps(readiness, indent=2))
assert readiness_response.status_code == 200
assert "backend_healthy" in readiness

cached_response = httpx.get(f"{BASE_URL}/health")
print("\nCached readiness status:", cached_response.status_code)
cached = cached_response.json()
print(json.dumps(cached, indent=2))
assert cached_response.status_code == 200

if readiness.get("backend") == "ipfs_cluster":
    assert readiness.get("add_mode") == "cluster_proxy"
    assert "available_cluster_peers" in readiness
    assert cached.get("cached") in {True, False}

print("\n✅ Liveness/readiness checks passed!")


## 2. Store JSON Content

In [ ]:
json_content = {
    "title": "Test Document",
    "author": "dARK Project",
    "description": "Integration test from notebook",
    "timestamp": "2026-02-08"
}

response = httpx.post(
    f"{BASE_URL}/v1/store",
    content=json.dumps(json_content),
    headers={"Content-Type": "application/json"}
)

print(f"Status: {response.status_code}")
store_result = response.json()
print(json.dumps(store_result, indent=2))

json_cid = store_result["cid"]

## 3. Retrieve JSON Content

In [ ]:
response = httpx.get(f"{BASE_URL}/v1/retrieve/{json_cid}")

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type')}")
print(json.dumps(response.json(), indent=2))

# Verify content matches
assert response.json() == json_content, "Content mismatch!"
print("\n✅ Content verification passed!")

## 4. Store XML Content

In [ ]:
xml_content = """<?xml version="1.0" encoding="UTF-8"?>
<metadata xmlns:dc="http://purl.org/dc/elements/1.1/">
    <dc:title>Test Document</dc:title>
    <dc:creator>dARK Project</dc:creator>
    <dc:date>2026-02-08</dc:date>
</metadata>"""

response = httpx.post(
    f"{BASE_URL}/v1/store",
    content=xml_content,
    headers={"Content-Type": "text/xml"}
)

print(f"Status: {response.status_code}")
xml_result = response.json()
print(json.dumps(xml_result, indent=2))

xml_cid = xml_result["cid"]

## 5. Retrieve XML Content

In [ ]:
response = httpx.get(f"{BASE_URL}/v1/retrieve/{xml_cid}")

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type')}")
print(response.text)

print("\n✅ XML retrieval passed!")

## 6. Check Pin Status

In [ ]:
response = httpx.get(f"{BASE_URL}/v1/status/{json_cid}")

print(f"JSON CID Status:")
print(json.dumps(response.json(), indent=2))

response = httpx.get(f"{BASE_URL}/v1/status/{xml_cid}")

print(f"\nXML CID Status:")
print(json.dumps(response.json(), indent=2))

## 7. Content-Addressable Verification

Same content should produce the same CID.

In [ ]:
# Store the same JSON content again
response = httpx.post(
    f"{BASE_URL}/v1/store",
    content=json.dumps(json_content),
    headers={"Content-Type": "application/json"}
)

new_cid = response.json()["cid"]

print(f"Original CID: {json_cid}")
print(f"New CID:      {new_cid}")

assert json_cid == new_cid, "CIDs don't match - content-addressable storage broken!"
print("\n✅ Content-addressable verification passed!")

## 8. Error Handling

In [ ]:
# Test 404 for non-existent CID
response = httpx.get(f"{BASE_URL}/v1/retrieve/nonexistentcid123")
print(f"Retrieve non-existent: {response.status_code}")
assert response.status_code == 404

# Test 400 for empty body
response = httpx.post(
    f"{BASE_URL}/v1/store",
    content="",
    headers={"Content-Type": "application/json"}
)
print(f"Store empty body: {response.status_code}")
assert response.status_code == 400

print("\n✅ Error handling tests passed!")

## Summary

All tests completed successfully!

In [ ]:
print("="*50)
print("dark-store-api Test Summary")
print("="*50)
print("✅ Liveness check")
print("✅ Readiness check")
print(f"✅ Store JSON -> CID: {json_cid[:20]}...")
print(f"✅ Retrieve JSON")
print(f"✅ Store XML -> CID: {xml_cid[:20]}...")
print(f"✅ Retrieve XML")
print(f"✅ Content-addressable verification")
print(f"✅ Error handling")
print("="*50)
print("All tests passed!")
